# 13 — OLID-I WL Kernel Construction

**Project:** Graph-Based Representation of Plant Leaf Images for Fine-Grained Plant Stress Classification Using Weisfeiler–Lehman Kernels
**Dataset:** OLID-I (4,387 accepted images · 57 classes · 8 crops · 20 conditions)

## Purpose
This notebook constructs and validates the **Weisfeiler–Lehman Gram kernel matrices** from the fold-safe WL subtree feature representations produced by Notebook 12.

**No SVM, CNN, GNN, accuracy, or Macro-F1 is computed here.**

The output is 100 validated kernel-fold configurations (5 graph types × 4 WL heights × 5 CV folds)
ready for Notebook 14 (SVM classification).

## Kernel Formulation
Given explicit WL subtree feature representations:

- Φ_train ∈ ℝ^(n_train × d)
- Φ_val ∈ ℝ^(n_val × d)

We construct:

- **K_train = Φ_train Φ_train^T** — positive semidefinite Gram matrix (n_train × n_train)
- **K_val = Φ_val Φ_train^T** — inductive validation kernel (n_val × n_train)

Because K_train is a Gram matrix of an explicit feature map, it is mathematically guaranteed to be PSD.


## 1. Imports & Configuration

In [1]:
import os
import gc
import json
import time
import warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
from collections import Counter, defaultdict
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings('ignore')

# ── Research constants ─────────────────────────────────────
RANDOM_STATE    = 42
N_BINS          = 3
BIN_LABELS      = ['Low', 'Medium', 'High']
CORR_THRESH     = 0.70
KNN_K           = 3
WL_HEIGHTS      = [0, 1, 2, 3]
N_FOLDS         = 5
PSD_TOL         = -1e-8           # eigenvalues below this threshold → FAIL
SYMM_TOL        = 1e-8            # asymmetry tolerance
SAVE_KERNELS    = True            # toggle to disable saving if disk is limited

# ── Paths ──────────────────────────────────────────────────
RESULTS_DIR    = 'results/wl_kernels'
SUMMARY_DIR    = 'results/wl_kernel_summary'
os.makedirs(RESULTS_DIR,  exist_ok=True)
os.makedirs(SUMMARY_DIR,  exist_ok=True)

print("Configuration:")
print(f"  RANDOM_STATE = {RANDOM_STATE}")
print(f"  N_BINS       = {N_BINS}")
print(f"  CORR_THRESH  = {CORR_THRESH}")
print(f"  KNN_K        = {KNN_K}")
print(f"  WL_HEIGHTS   = {WL_HEIGHTS}")
print(f"  N_FOLDS      = {N_FOLDS}")
print(f"  PSD_TOL      = {PSD_TOL}")
print(f"  SYMM_TOL     = {SYMM_TOL}")
print(f"  SAVE_KERNELS = {SAVE_KERNELS}")
print(f"  RESULTS_DIR  = {RESULTS_DIR}")


Configuration:
  RANDOM_STATE = 42
  N_BINS       = 3
  CORR_THRESH  = 0.7
  KNN_K        = 3
  WL_HEIGHTS   = [0, 1, 2, 3]
  N_FOLDS      = 5
  PSD_TOL      = -1e-08
  SYMM_TOL     = 1e-08
  SAVE_KERNELS = True
  RESULTS_DIR  = results/wl_kernels


## 2. Data Loading & Validation

In [2]:
csv_path    = 'olid_full_features.csv'
df_full     = pd.read_csv(csv_path)
df_accepted = df_full[df_full['quality_flag'] == 'ACCEPT'].copy().reset_index(drop=True)

meta_cols    = ['image_path', 'image_name', 'class', 'crop', 'condition',
                'quality_flag', 'mask_area_ratio', 'failure_reason']
feature_cols = [c for c in df_accepted.columns if c not in meta_cols]

assert len(df_accepted)  == 4387, f"Expected 4387 accepted images, got {len(df_accepted)}"
assert len(feature_cols) == 38,   f"Expected 38 features, got {len(feature_cols)}"
assert df_accepted[feature_cols].isna().sum().sum() == 0
assert np.isinf(df_accepted[feature_cols].values).sum() == 0

print(f"Accepted images : {len(df_accepted)}")
print(f"Feature columns : {len(feature_cols)}")
print(f"Classes         : {df_accepted['class'].nunique()}")
print(f"Data checks     : PASS")


Accepted images : 4387
Feature columns : 38
Classes         : 57
Data checks     : PASS


## 3. Feature Definitions (from Notebook 11)

In [3]:
SHAPE_FEATURES         = ['area', 'perimeter', 'aspect_ratio', 'circularity', 'extent',
                           'solidity', 'convex_area', 'major_axis_length', 'minor_axis_length', 'eccentricity']
COLOR_RGB_FEATURES     = ['mean_R', 'std_R', 'mean_G', 'std_G', 'mean_B', 'std_B']
COLOR_HSV_FEATURES     = ['mean_H', 'std_H', 'mean_S', 'std_S', 'mean_V', 'std_V']
COLOR_LAB_FEATURES     = ['mean_L', 'std_L', 'mean_a', 'std_a', 'mean_b', 'std_b']
DERIVED_COLOR_FEATURES = ['green_ratio', 'yellow_ratio']
TEXTURE_FEATURES       = ['glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']
SPATIAL_FEATURES       = ['core_mean_H', 'core_yellow_ratio', 'edge_mean_H', 'edge_yellow_ratio']

FEATURE_FAMILIES = {
    'Shape':        SHAPE_FEATURES,
    'RGB':          COLOR_RGB_FEATURES,
    'HSV':          COLOR_HSV_FEATURES,
    'Lab':          COLOR_LAB_FEATURES,
    'DerivedColor': DERIVED_COLOR_FEATURES,
    'Texture':      TEXTURE_FEATURES,
    'Spatial':      SPATIAL_FEATURES,
}
print("Feature families loaded.")


Feature families loaded.


## 4. WL Engine (Reused from Notebook 12)

Full implementations of `FoldSafeDiscretizer`, graph constructors, `WLLabelMapper`,
and `wl_subtree_features()` are reproduced here to make this notebook standalone
and reproducible without requiring Notebook 12 to be open.


In [4]:
# ─── FoldSafeDiscretizer ───────────────────────────────────────────────────
class FoldSafeDiscretizer:
    def __init__(self, n_bins=3, strategy='quantile'):
        self.discretizer = KBinsDiscretizer(
            n_bins=n_bins, encode='ordinal', strategy=strategy, subsample=None
        )
        self.labels    = ['Low', 'Medium', 'High']
        self.is_fitted = False

    def fit(self, X_train):
        self.columns_ = list(X_train.columns)
        self.discretizer.fit(X_train.values)
        self.is_fitted = True
        return self

    def transform(self, X):
        if not self.is_fitted:
            raise ValueError("Must call fit() first.")
        X_disc = self.discretizer.transform(X.values).astype(int)
        label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
        df_lbl = pd.DataFrame(X_disc, columns=list(X.columns), index=X.index)
        for col in df_lbl.columns:
            df_lbl[col] = df_lbl[col].map(label_map)
        assert df_lbl.isna().sum().sum() == 0
        return df_lbl


# ─── GraphConstructor base ────────────────────────────────────────────────
class GraphConstructor:
    def __init__(self, features):
        self.features   = list(features)
        self.base_graph = nx.Graph()
        self.base_graph.add_nodes_from(self.features)

    def fit(self, X_train): raise NotImplementedError

    def build_edges(self): return list(self.base_graph.edges(data=True))

    def transform_single(self, label_row):
        G = self.base_graph.copy()
        for node in self.features:
            G.nodes[node]['label'] = label_row[node]
        return G

    def transform(self, df_labels):
        graphs = []
        for _, row in df_labels.iterrows():
            g = self.transform_single(row)
            if g.number_of_edges() == 0:
                warnings.warn(f"Zero edges for index {row.name}.", RuntimeWarning)
            graphs.append(g)
        return graphs


class CorrelationGraph(GraphConstructor):
    def __init__(self, features, threshold=0.70):
        super().__init__(features)
        self.threshold = threshold

    def fit(self, X_train):
        corr = X_train[self.features].corr(method='pearson')
        self.base_graph.clear_edges()
        n = len(self.features)
        for i in range(n):
            for j in range(i+1, n):
                if abs(corr.iloc[i, j]) >= self.threshold:
                    self.base_graph.add_edge(self.features[i], self.features[j],
                                             weight=float(corr.iloc[i, j]), source='correlation')
        return self


class FeatureSimilarityKNNGraph(GraphConstructor):
    def __init__(self, features, k=3):
        super().__init__(features)
        self.k      = k
        self.scaler = StandardScaler()

    def fit(self, X_train):
        X_scaled = self.scaler.fit_transform(X_train[self.features])
        X_t      = X_scaled.T
        nn = NearestNeighbors(n_neighbors=self.k + 1, metric='euclidean')
        nn.fit(X_t)
        _, indices = nn.kneighbors(X_t)
        self.base_graph.clear_edges()
        for i, nbrs in enumerate(indices):
            for j in nbrs[1:]:
                self.base_graph.add_edge(self.features[i], self.features[j],
                                         source='feature_similarity')
        return self


class FeatureFamilyGraph(GraphConstructor):
    def __init__(self, features, families):
        super().__init__(features)
        self.families = families

    def fit(self, X_train):
        self.base_graph.clear_edges()
        corr = X_train[self.features].corr().abs()
        for fam_feats in self.families.values():
            vf = [f for f in fam_feats if f in self.features]
            for i in range(len(vf)):
                for j in range(i+1, len(vf)):
                    self.base_graph.add_edge(vf[i], vf[j], source='family')
        fams = list(self.families.keys())
        for i in range(len(fams)):
            for j in range(i+1, len(fams)):
                fa = [f for f in self.families[fams[i]] if f in self.features]
                fb = [f for f in self.families[fams[j]] if f in self.features]
                if fa and fb:
                    sub = corr.loc[fa, fb]
                    mx = sub.max().max()
                    if pd.notna(mx):
                        ri, ci = np.where(sub.values == mx)
                        self.base_graph.add_edge(fa[ri[0]], fb[ci[0]],
                                                 source='family_bridge', weight=float(mx))
        return self


class SpatialGraph(GraphConstructor):
    def __init__(self, features, spatial_features, corr_threshold=0.60):
        super().__init__(features)
        self.spatial_features = spatial_features
        self.corr_threshold   = corr_threshold

    def fit(self, X_train):
        self.base_graph.clear_edges()
        core_feats   = [f for f in self.spatial_features if f.startswith('core_')]
        edge_feats   = [f for f in self.spatial_features if f.startswith('edge_')]
        global_feats = [f for f in self.features if f not in self.spatial_features]
        corr         = X_train[self.features].corr().abs()
        for i in range(len(core_feats)):
            for j in range(i+1, len(core_feats)):
                self.base_graph.add_edge(core_feats[i], core_feats[j], source='spatial_core')
        for i in range(len(edge_feats)):
            for j in range(i+1, len(edge_feats)):
                self.base_graph.add_edge(edge_feats[i], edge_feats[j], source='spatial_edge')
        for cf in core_feats:
            ef = 'edge_' + cf.replace('core_', '')
            if ef in edge_feats:
                self.base_graph.add_edge(cf, ef, source='spatial_bridge')
        for gf in global_feats:
            for sf in self.spatial_features:
                if corr.loc[gf, sf] >= self.corr_threshold:
                    self.base_graph.add_edge(gf, sf, source='spatial_global_link',
                                             weight=float(corr.loc[gf, sf]))
        return self


class HybridGraph(GraphConstructor):
    def __init__(self, features, corr_threshold=0.80, knn_k=3,
                 families=None, spatial_features=None, spatial_threshold=0.70):
        super().__init__(features)
        self._corr_t = corr_threshold
        self._knn_k  = knn_k
        self._fams   = families or FEATURE_FAMILIES
        self._spatf  = spatial_features or SPATIAL_FEATURES
        self._spat_t = spatial_threshold

    def fit(self, X_train):
        self.base_graph.clear_edges()
        subs = [
            (CorrelationGraph(self.features, self._corr_t).fit(X_train), 'correlation'),
            (FeatureFamilyGraph(self.features, self._fams).fit(X_train), 'family'),
            (SpatialGraph(self.features, self._spatf, self._spat_t).fit(X_train), 'spatial'),
        ]
        for sub, src in subs:
            for u, v, d in sub.base_graph.edges(data=True):
                if self.base_graph.has_edge(u, v):
                    cur = self.base_graph[u][v].get('source', '')
                    if src not in cur:
                        self.base_graph[u][v]['source'] = 'multiple'
                else:
                    self.base_graph.add_edge(u, v, source=src)
        return self


GRAPH_BUILDERS = {
    'correlation': lambda f: CorrelationGraph(f, CORR_THRESH),
    'knn':         lambda f: FeatureSimilarityKNNGraph(f, KNN_K),
    'family':      lambda f: FeatureFamilyGraph(f, FEATURE_FAMILIES),
    'spatial':     lambda f: SpatialGraph(f, SPATIAL_FEATURES),
    'hybrid':      lambda f: HybridGraph(f),
}
GRAPH_NAMES = {
    'correlation': 'Correlation',
    'knn':         'Feature-Similarity kNN',
    'family':      'Feature-Family',
    'spatial':     'Spatial/Regional',
    'hybrid':      'Hybrid',
}
print("Graph constructors loaded:", list(GRAPH_BUILDERS.keys()))


Graph constructors loaded: ['correlation', 'knn', 'family', 'spatial', 'hybrid']


In [5]:
# ─── WLLabelMapper ────────────────────────────────────────────────────────
class WLLabelMapper:
    def __init__(self):
        self._maps    = {}
        self._counter = {}
        self._oov     = {}

    def _key(self, sig):
        label, neighbors = sig
        return f"{label}|{','.join(sorted(neighbors))}"

    def map_training_signature(self, iteration, sig):
        if iteration not in self._maps:
            self._maps[iteration]    = {}
            self._counter[iteration] = 0
            self._oov[iteration]     = set()
        key = self._key(sig)
        if key not in self._maps[iteration]:
            self._maps[iteration][key] = f"wl_{iteration}_{self._counter[iteration]}"
            self._counter[iteration]  += 1
        return self._maps[iteration][key]

    def map_inference_signature(self, iteration, sig):
        if iteration not in self._maps:
            return f"__OOV__{iteration}"
        key = self._key(sig)
        if key not in self._maps[iteration]:
            self._oov[iteration].add(key)
            return f"__OOV__{iteration}"
        return self._maps[iteration][key]


# ─── WL Subtree Feature Engine ────────────────────────────────────────────
def wl_subtree_features(graphs_train, graphs_val, max_height=3):
    mapper = WLLabelMapper()
    train_node_lbl = [{n: G.nodes[n]['label'] for n in G.nodes()} for G in graphs_train]
    val_node_lbl   = [{n: G.nodes[n]['label'] for n in G.nodes()} for G in graphs_val]
    train_bags = [Counter() for _ in graphs_train]
    val_bags   = [Counter() for _ in graphs_val]

    def _iterate(graphs, node_lbls, iteration, training):
        new_lbls = []
        for g_idx, G in enumerate(graphs):
            cur = node_lbls[g_idx]
            new = {}
            for node in G.nodes():
                nbr = sorted([cur[n] for n in G.neighbors(node)])
                sig = (cur[node], tuple(nbr))
                new[node] = (mapper.map_training_signature(iteration, sig) if training
                             else mapper.map_inference_signature(iteration, sig))
            new_lbls.append(new)
        return new_lbls

    for g_idx in range(len(graphs_train)):
        for lbl in train_node_lbl[g_idx].values():
            train_bags[g_idx][lbl] += 1
    for g_idx in range(len(graphs_val)):
        for lbl in val_node_lbl[g_idx].values():
            val_bags[g_idx][lbl] += 1

    for h in range(1, max_height + 1):
        train_node_lbl = _iterate(graphs_train, train_node_lbl, h-1, True)
        val_node_lbl   = _iterate(graphs_val,   val_node_lbl,   h-1, False)
        for g_idx in range(len(graphs_train)):
            for lbl in train_node_lbl[g_idx].values():
                train_bags[g_idx][lbl] += 1
        for g_idx in range(len(graphs_val)):
            for lbl in val_node_lbl[g_idx].values():
                if not lbl.startswith('__OOV__'):
                    val_bags[g_idx][lbl] += 1

    all_labels   = set()
    for bag in train_bags:
        all_labels.update(bag.keys())
    feature_index = {lbl: idx for idx, lbl in enumerate(sorted(all_labels))}
    n_feat        = len(feature_index)

    def _to_sparse(bags, fi, n_f):
        rows, cols, vals = [], [], []
        for g_idx, bag in enumerate(bags):
            for lbl, cnt in bag.items():
                if lbl in fi:
                    rows.append(g_idx); cols.append(fi[lbl]); vals.append(cnt)
        return sp.csr_matrix((vals, (rows, cols)), shape=(len(bags), n_f))

    Phi_tr = _to_sparse(train_bags, feature_index, n_feat)
    Phi_va = _to_sparse(val_bags,   feature_index, n_feat)
    return Phi_tr, Phi_va, feature_index


# ─── Full Fold Pipeline ───────────────────────────────────────────────────
def build_wl_representations(X_tr, X_va, gt_key, wl_height):
    """Fit graph + discretizer on X_tr, build WL Phi matrices."""
    constructor = GRAPH_BUILDERS[gt_key](feature_cols)
    constructor.fit(X_tr)
    disc = FoldSafeDiscretizer(n_bins=N_BINS)
    disc.fit(X_tr)
    df_tr = disc.transform(X_tr)
    df_va = disc.transform(X_va)
    g_tr  = constructor.transform(df_tr)
    g_va  = constructor.transform(df_va)
    Phi_tr, Phi_va, fi = wl_subtree_features(g_tr, g_va, max_height=wl_height)
    return Phi_tr, Phi_va, fi

print("WL engine (NB12 reuse) loaded successfully.")


WL engine (NB12 reuse) loaded successfully.


## 5. Kernel Construction Functions

### Mathematical Basis

For explicit feature vectors:

**k(G, G') = ⟨Φ(G), Φ(G')⟩**

therefore:

**K_train = Φ_train Φ_train^T** — PSD Gram matrix

**K_val = Φ_val Φ_train^T** — inductive cross-kernel (n_val × n_train)

Both operate on sparse Φ matrices and produce dense kernel matrices.


In [6]:
def build_kernel(Phi_train, Phi_val=None):
    """
    Construct WL subtree kernel matrices.

    K_train = Phi_train @ Phi_train.T   shape: (n_train, n_train)
    K_val   = Phi_val   @ Phi_train.T   shape: (n_val, n_train)

    Both Phi arguments may be scipy sparse or dense numpy arrays.
    Returns dense numpy arrays.
    """
    if sp.issparse(Phi_train):
        K_train = (Phi_train @ Phi_train.T).toarray().astype(np.float64)
    else:
        K_train = (Phi_train @ Phi_train.T).astype(np.float64)

    K_val = None
    if Phi_val is not None:
        if sp.issparse(Phi_val):
            K_val = (Phi_val @ Phi_train.T).toarray().astype(np.float64)
        else:
            K_val = (Phi_val @ Phi_train.T).astype(np.float64)

    return K_train, K_val


def normalize_kernel(K_train, K_val=None):
    """
    Compute the normalized WL kernel.

    K_norm(i,j) = K(i,j) / sqrt(K(i,i) * K(j,j))

    For the cross-kernel K_val (rows=val, cols=train):
        K_val_norm(i,j) = K_val(i,j) / sqrt(K_val_self(i,i) * K_train(j,j))
    where K_val_self = Phi_val @ Phi_val.T is computed internally and is
    NEVER used for SVM prediction (only for normalization bookkeeping).

    Handles zero diagonal safely (result is 0 for those entries).
    """
    diag_tr = np.diag(K_train)
    # Guard: replace exact zeros with 1 to avoid divide-by-zero (result stays 0)
    safe_diag_tr = np.where(diag_tr > 0, diag_tr, 1.0)
    norm_train   = np.sqrt(np.outer(safe_diag_tr, safe_diag_tr))
    K_train_norm = K_train / norm_train
    # Zero out entries where original diagonal was zero
    zero_mask    = (diag_tr == 0)
    K_train_norm[zero_mask, :] = 0
    K_train_norm[:, zero_mask] = 0

    K_val_norm = None
    if K_val is not None:
        # This requires the validation self-kernel diagonal for correct normalization.
        # K_val_self is computed ONLY for normalization — it is NOT used in SVM prediction.
        raise ValueError(
            "normalize_kernel() for K_val requires Phi_val. "
            "Call normalize_kernel_val(K_train, K_val, Phi_val) instead."
        )
    return K_train_norm, None


def normalize_kernel_val(K_train, K_val, Phi_val):
    """
    Normalize the cross-kernel K_val correctly.

    K_val_norm(i,j) = K_val(i,j) / sqrt(K_val_self(i,i) * K_train(j,j))

    K_val_self = Phi_val @ Phi_val.T is computed here ONLY for normalization.
    It is NOT returned and NOT used for SVM prediction.
    """
    diag_tr = np.diag(K_train)
    # Compute Phi_val self-kernel diagonal only (not full matrix)
    if sp.issparse(Phi_val):
        # row-wise squared norms: sum of squared entries per row
        val_self_diag = np.asarray(Phi_val.multiply(Phi_val).sum(axis=1)).ravel().astype(np.float64)
    else:
        val_self_diag = np.sum(Phi_val ** 2, axis=1).astype(np.float64)

    safe_val  = np.where(val_self_diag > 0, val_self_diag, 1.0)
    safe_tr   = np.where(diag_tr > 0, diag_tr, 1.0)
    norm_cross = np.sqrt(np.outer(safe_val, safe_tr))
    K_val_norm = K_val / norm_cross

    # Zero out rows/cols where diagonal was zero
    zero_val = (val_self_diag == 0)
    zero_tr  = (diag_tr == 0)
    K_val_norm[zero_val, :] = 0
    K_val_norm[:, zero_tr]  = 0
    return K_val_norm


print("build_kernel(), normalize_kernel(), normalize_kernel_val() defined.")


build_kernel(), normalize_kernel(), normalize_kernel_val() defined.


## 6. Kernel Validation Functions

In [7]:
def check_finite(K, name='K'):
    """Check no NaN or Inf values."""
    has_nan = np.isnan(K).any()
    has_inf = np.isinf(K).any()
    status  = 'PASS' if not (has_nan or has_inf) else 'FAIL'
    return {'name': name, 'has_nan': has_nan, 'has_inf': has_inf, 'status': status}


def check_symmetry(K, tol=SYMM_TOL, name='K_train'):
    """Check K ≈ K^T within tolerance."""
    diff      = np.abs(K - K.T)
    max_asym  = float(diff.max())
    is_symm   = max_asym <= tol
    status    = 'PASS' if is_symm else 'FAIL'
    return {'name': name, 'max_asymmetry': max_asym, 'tolerance': tol, 'status': status}


def check_psd(K, tol=PSD_TOL, name='K_train', max_eig_samples=None):
    """
    Check positive semidefiniteness via eigenvalue decomposition.
    For large matrices (n > 1000), sample a random submatrix for speed,
    document that the full matrix is a Gram matrix by construction, and
    report accordingly.
    """
    n = K.shape[0]
    if max_eig_samples is not None and n > max_eig_samples:
        rng   = np.random.default_rng(RANDOM_STATE)
        idx   = rng.choice(n, size=max_eig_samples, replace=False)
        K_sub = K[np.ix_(idx, idx)]
        note  = f'submatrix({max_eig_samples}x{max_eig_samples})'
    else:
        K_sub = K
        note  = 'full'

    eigs      = np.linalg.eigvalsh(K_sub)
    min_eig   = float(eigs.min())
    max_eig   = float(eigs.max())
    n_neg     = int((eigs < tol).sum())
    is_psd    = min_eig >= tol
    status    = 'PASS' if is_psd else 'FAIL'
    return {
        'name': name, 'min_eigenvalue': min_eig, 'max_eigenvalue': max_eig,
        'n_below_tol': n_neg, 'tolerance': tol, 'status': status,
        'note': f'Gram matrix by construction. Eigenvalue check on {note}.'
    }


def check_diagonal(K, name='K_train'):
    """Check K[i,i] >= 0 and report zero-diagonal entries."""
    diag    = np.diag(K)
    min_d   = float(diag.min())
    max_d   = float(diag.max())
    n_zero  = int((diag == 0).sum())
    n_neg   = int((diag < 0).sum())
    status  = 'PASS' if n_neg == 0 else 'FAIL'
    return {'name': name, 'min_diag': min_d, 'max_diag': max_d,
            'n_zero_diag': n_zero, 'n_neg_diag': n_neg, 'status': status}


def check_shape(K_train, K_val, Phi_train, Phi_val):
    """Verify matrix dimensions are consistent."""
    n_tr, d = Phi_train.shape
    n_va    = Phi_val.shape[0]
    checks  = {
        'K_train_shape_ok':  K_train.shape == (n_tr, n_tr),
        'K_val_shape_ok':    K_val.shape   == (n_va, n_tr),
        'K_val_cols_eq_tr':  K_val.shape[1] == K_train.shape[0],
    }
    status = 'PASS' if all(checks.values()) else 'FAIL'
    return {**checks, 'status': status,
            'K_train_shape': K_train.shape, 'K_val_shape': K_val.shape}


def run_all_kernel_checks(K_train, K_val, Phi_train, Phi_val,
                          name_prefix='', max_eig_samples=500):
    """Aggregate all checks. Returns a dict of results."""
    r = {}
    r['finite_train']  = check_finite(K_train, name_prefix + 'K_train')
    r['finite_val']    = check_finite(K_val,   name_prefix + 'K_val')
    r['symmetry']      = check_symmetry(K_train, name=name_prefix + 'K_train')
    r['psd']           = check_psd(K_train, name=name_prefix + 'K_train',
                                   max_eig_samples=max_eig_samples)
    r['diagonal']      = check_diagonal(K_train, name=name_prefix + 'K_train')
    r['shape']         = check_shape(K_train, K_val, Phi_train, Phi_val)
    overall = all(v.get('status') == 'PASS' for v in r.values())
    r['overall_pass']  = overall
    return r


print("Kernel validation functions defined.")


Kernel validation functions defined.


## 7. Manual Kernel Example (Hand-Checkable)

Verify the kernel construction with a tiny feature matrix where the answer
can be computed by hand.

```
Φ = [[1, 2],
     [3, 4],
     [0, 1]]

K = Φ Φ^T =
[[1*1+2*2, 1*3+2*4, 1*0+2*1],
 [3*1+4*2, 3*3+4*4, 3*0+4*1],
 [0*1+1*2, 0*3+1*4, 0*0+1*1]]
=
[[5,  11, 2],
 [11, 25, 4],
 [2,   4, 1]]
```


In [8]:
Phi_toy   = np.array([[1.0, 2.0], [3.0, 4.0], [0.0, 1.0]])
K_expected = np.array([[5., 11., 2.], [11., 25., 4.], [2., 4., 1.]])

K_computed, _ = build_kernel(Phi_toy)
assert np.allclose(K_computed, K_expected, atol=1e-12), "Manual kernel example FAILED!"

print("Manual Kernel Example")
print("=====================")
print("Phi (feature matrix):")
print(Phi_toy)
print()
print("Expected K = Phi @ Phi.T:")
print(K_expected)
print()
print("Computed K:")
print(K_computed)
print()
print(f"Max absolute difference: {np.abs(K_computed - K_expected).max():.2e}")
print("Manual kernel example: PASS")


Manual Kernel Example
Phi (feature matrix):
[[1. 2.]
 [3. 4.]
 [0. 1.]]

Expected K = Phi @ Phi.T:
[[ 5. 11.  2.]
 [11. 25.  4.]
 [ 2.  4.  1.]]

Computed K:
[[ 5. 11.  2.]
 [11. 25.  4.]
 [ 2.  4.  1.]]

Max absolute difference: 0.00e+00
Manual kernel example: PASS


## 8. Cross-Validation Fold Setup

Using the same 5-fold stratified CV as Notebooks 07 and 12 (RANDOM_STATE=42).


In [9]:
X_all = df_accepted[feature_cols]
y_all = df_accepted['class']

skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(X_all, y_all))

print(f"StratifiedKFold: {N_FOLDS} folds, random_state={RANDOM_STATE}")
for fold_idx, (tr_idx, va_idx) in enumerate(folds):
    print(f"  Fold {fold_idx}: train={len(tr_idx)}, val={len(va_idx)}")


StratifiedKFold: 5 folds, random_state=42
  Fold 0: train=3509, val=878
  Fold 1: train=3509, val=878
  Fold 2: train=3510, val=877
  Fold 3: train=3510, val=877
  Fold 4: train=3510, val=877


## 9. Sanity Check — One Configuration

Before the full run, validate one complete configuration:
**Correlation graph · h=1 · Fold 0**


In [10]:
print("SANITY CHECK: Correlation graph, h=1, Fold 0")
print("=" * 50)

tr_idx, va_idx = folds[0]
X_tr = X_all.iloc[tr_idx]
X_va = X_all.iloc[va_idx]

t0 = time.time()
Phi_tr, Phi_va, fi = build_wl_representations(X_tr, X_va, 'correlation', 1)
t_wl = time.time() - t0

t1 = time.time()
K_train, K_val = build_kernel(Phi_tr, Phi_va)
t_k = time.time() - t1

print(f"Phi_train : {Phi_tr.shape}  nnz={Phi_tr.nnz}")
print(f"Phi_val   : {Phi_va.shape}  nnz={Phi_va.nnz}")
print(f"K_train   : {K_train.shape}")
print(f"K_val     : {K_val.shape}")
print(f"WL time   : {t_wl:.2f}s")
print(f"Kernel time: {t_k:.3f}s")
print()

checks = run_all_kernel_checks(K_train, K_val, Phi_tr, Phi_va, max_eig_samples=400)
for k, v in checks.items():
    if isinstance(v, dict):
        print(f"  {k}: {v.get('status', '')}  ({', '.join(f'{a}={b}' for a,b in v.items() if a != 'status' and a != 'name' and a != 'note')})")
    else:
        print(f"  overall_pass: {v}")

print()
assert checks['overall_pass'], "SANITY CHECK FAILED — investigate before full run!"
print("Sanity check: PASS — proceeding to full run.")

# Kernel orientation check
assert K_val.shape[1] == K_train.shape[0], "K_val column count != K_train row count!"
print("Validation-to-training kernel orientation: PASS")

# Determinism check
Phi_tr2, Phi_va2, fi2 = build_wl_representations(X_tr, X_va, 'correlation', 1)
K_train2, K_val2 = build_kernel(Phi_tr2, Phi_va2)
assert np.allclose(K_train, K_train2, atol=1e-12), "K_train not deterministic!"
assert np.allclose(K_val, K_val2, atol=1e-12), "K_val not deterministic!"
print("Determinism (two runs): PASS")

del K_train, K_val, K_train2, K_val2, Phi_tr2, Phi_va2
gc.collect()


SANITY CHECK: Correlation graph, h=1, Fold 0
Phi_train : (3509, 455)  nnz=91395
Phi_val   : (878, 455)  nnz=22853
K_train   : (3509, 3509)
K_val     : (878, 3509)
WL time   : 1.37s
Kernel time: 0.294s

  finite_train: PASS  (has_nan=False, has_inf=False)
  finite_val: PASS  (has_nan=False, has_inf=False)
  symmetry: PASS  (max_asymmetry=0.0, tolerance=1e-08)
  psd: PASS  (min_eigenvalue=-1.998833031716457e-11, max_eigenvalue=201095.7705967842, n_below_tol=0, tolerance=-1e-08)
  diagonal: PASS  (min_diag=552.0, max_diag=1332.0, n_zero_diag=0, n_neg_diag=0)
  shape: PASS  (K_train_shape_ok=True, K_val_shape_ok=True, K_val_cols_eq_tr=True, K_train_shape=(3509, 3509), K_val_shape=(878, 3509))
  overall_pass: True

Sanity check: PASS — proceeding to full run.
Validation-to-training kernel orientation: PASS
Determinism (two runs): PASS


175480

## 10. Normalization Check on Sanity Fold

In [11]:
# Rebuild for normalisation check
Phi_tr_n, Phi_va_n, _ = build_wl_representations(X_tr, X_va, 'correlation', 1)
K_tr_raw, K_va_raw = build_kernel(Phi_tr_n, Phi_va_n)

K_tr_norm = normalize_kernel(K_tr_raw)[0]
K_va_norm = normalize_kernel_val(K_tr_raw, K_va_raw, Phi_va_n)

diag_norm = np.diag(K_tr_norm)
print("Normalized Kernel — Training Diagonal (should all ≈ 1.0):")
print(f"  min: {diag_norm.min():.6f}  max: {diag_norm.max():.6f}  mean: {diag_norm.mean():.6f}")
all_one = np.allclose(diag_norm, 1.0, atol=1e-8)
print(f"  All diag(K_train_norm) ≈ 1.0: {all_one}  ({'PASS' if all_one else 'INVESTIGATE'})")

# K_val_norm: rows = validation, cols = training -> diagonal is NOT expected to be 1
print()
print("Normalized Cross-Kernel K_val_norm:")
print(f"  shape: {K_va_norm.shape}")
fin_check = check_finite(K_va_norm, 'K_val_norm')
print(f"  Finite values check: {fin_check['status']}")

del K_tr_raw, K_va_raw, K_tr_norm, K_va_norm, Phi_tr_n, Phi_va_n
gc.collect()
print()
print("Normalization check complete.")


Normalized Kernel — Training Diagonal (should all ≈ 1.0):
  min: 1.000000  max: 1.000000  mean: 1.000000
  All diag(K_train_norm) ≈ 1.0: True  (PASS)

Normalized Cross-Kernel K_val_norm:
  shape: (878, 3509)
  Finite values check: PASS

Normalization check complete.


## 11. Feature-to-Kernel Consistency Check

Verify that K_train = Φ_train Φ_train^T by independently computing a few
pairwise dot products and comparing against K.


In [12]:
Phi_ck, _, _ = build_wl_representations(X_tr, X_va, 'correlation', 1)
K_ck, _ = build_kernel(Phi_ck)

# Spot-check 5 random pairs
rng  = np.random.default_rng(RANDOM_STATE)
idx1 = rng.integers(0, Phi_ck.shape[0], size=5)
idx2 = rng.integers(0, Phi_ck.shape[0], size=5)

print("Feature-to-Kernel Consistency Check")
print("=====================================")
for i, j in zip(idx1, idx2):
    row_i = np.asarray(Phi_ck[i].todense()).ravel() if sp.issparse(Phi_ck) else Phi_ck[i]
    row_j = np.asarray(Phi_ck[j].todense()).ravel() if sp.issparse(Phi_ck) else Phi_ck[j]
    dot   = float(np.dot(row_i, row_j))
    k_ij  = float(K_ck[i, j])
    match = abs(dot - k_ij) < 1e-8
    print(f"  K[{i},{j}]: from Phi dot = {dot:.4f}, from K = {k_ij:.4f}, match = {match}")

assert all(abs(float(np.dot(
    np.asarray(Phi_ck[i].todense()).ravel() if sp.issparse(Phi_ck) else Phi_ck[i],
    np.asarray(Phi_ck[j].todense()).ravel() if sp.issparse(Phi_ck) else Phi_ck[j]
)) - float(K_ck[i, j])) < 1e-8 for i, j in zip(idx1, idx2)), "Consistency check FAILED!"
print()
print("Feature-to-Kernel consistency: PASS")

del Phi_ck, K_ck
gc.collect()


Feature-to-Kernel Consistency Check
  K[313,3012]: from Phi dot = 495.0000, from K = 495.0000, match = True
  K[2715,301]: from Phi dot = 588.0000, from K = 588.0000, match = True
  K[2296,2447]: from Phi dot = 492.0000, from K = 492.0000, match = True
  K[1540,706]: from Phi dot = 532.0000, from K = 532.0000, match = True
  K[1519,330]: from Phi dot = 489.0000, from K = 489.0000, match = True

Feature-to-Kernel consistency: PASS


175480

## 12. Full Kernel Construction Run

5 graph types × 4 WL heights × 5 CV folds = **100 kernel configurations**.

Each configuration is validated, optionally saved, then released from memory.


In [13]:
manifest_rows = []
validation_rows = []
runtime_rows = []
dim_rows = []

GRAPH_TYPES  = list(GRAPH_BUILDERS.keys())
total_expected = len(GRAPH_TYPES) * len(WL_HEIGHTS) * N_FOLDS
n_success, n_fail = 0, 0

print(f"Starting full kernel run: {total_expected} configurations")
print()

for gt_key in GRAPH_TYPES:
    for h in WL_HEIGHTS:
        for fold_idx, (tr_idx, va_idx) in enumerate(folds):
            cfg_label = f"{GRAPH_NAMES[gt_key]} | h={h} | fold={fold_idx}"
            try:
                X_tr_f = X_all.iloc[tr_idx]
                X_va_f = X_all.iloc[va_idx]
                n_tr   = len(tr_idx)
                n_va   = len(va_idx)

                # ── 1. WL feature representations ─────────────────────
                t_wl0 = time.time()
                Phi_tr, Phi_va, fi = build_wl_representations(X_tr_f, X_va_f, gt_key, h)
                t_wl = time.time() - t_wl0
                d = Phi_tr.shape[1]

                # ── 2. Raw kernel ──────────────────────────────────────
                t_k0 = time.time()
                K_train, K_val = build_kernel(Phi_tr, Phi_va)
                t_kernel = time.time() - t_k0

                # ── 3. Normalized kernel ───────────────────────────────
                t_n0 = time.time()
                K_train_norm = normalize_kernel(K_train)[0]
                K_val_norm   = normalize_kernel_val(K_train, K_val, Phi_va)
                t_norm = time.time() - t_n0

                # ── 4. Validate ────────────────────────────────────────
                chks = run_all_kernel_checks(K_train, K_val, Phi_tr, Phi_va,
                                             max_eig_samples=400)
                chks_norm_fin = check_finite(K_train_norm, 'K_train_norm')
                all_pass      = chks['overall_pass'] and chks_norm_fin['status'] == 'PASS'

                # ── 5. Save ────────────────────────────────────────────
                ktr_path = kva_path = ''
                if SAVE_KERNELS:
                    save_dir = os.path.join(RESULTS_DIR, gt_key, f'h{h}', f'fold{fold_idx}')
                    os.makedirs(save_dir, exist_ok=True)
                    ktr_path = os.path.join(save_dir, 'K_train.npz')
                    kva_path = os.path.join(save_dir, 'K_val.npz')
                    ktr_norm_path = os.path.join(save_dir, 'K_train_norm.npz')
                    kva_norm_path = os.path.join(save_dir, 'K_val_norm.npz')
                    np.savez_compressed(ktr_path, K=K_train,      n_tr=n_tr, n_va=n_va, d=d)
                    np.savez_compressed(kva_path, K=K_val,        n_tr=n_tr, n_va=n_va, d=d)
                    np.savez_compressed(ktr_norm_path, K=K_train_norm)
                    np.savez_compressed(kva_norm_path, K=K_val_norm)
                    meta = {
                        'task': '57class', 'graph_type': gt_key, 'wl_height': h,
                        'fold': fold_idx, 'n_train': n_tr, 'n_val': n_va,
                        'wl_feature_dim': d, 'normalization': 'both', 'dtype': 'float64',
                        'random_state': RANDOM_STATE
                    }
                    with open(os.path.join(save_dir, 'meta.json'), 'w') as f:
                        json.dump(meta, f, indent=2)

                # ── 6. Record ──────────────────────────────────────────
                manifest_rows.append({
                    'task': '57class',
                    'graph_type': gt_key,
                    'wl_height': h,
                    'fold': fold_idx,
                    'K_train_path': ktr_path,
                    'K_val_path':   kva_path,
                    'shape_train':  f"{K_train.shape}",
                    'shape_val':    f"{K_val.shape}",
                    'normalization': 'raw+normalized',
                    'dtype':         'float64',
                })

                validation_rows.append({
                    'graph_type': GRAPH_NAMES[gt_key],
                    'wl_height': h,
                    'fold': fold_idx,
                    'n_train': n_tr,
                    'n_val': n_va,
                    'wl_dim': d,
                    'K_train_shape': f"{K_train.shape}",
                    'K_val_shape': f"{K_val.shape}",
                    'symmetry': chks['symmetry']['status'],
                    'psd': chks['psd']['status'],
                    'finite_train': chks['finite_train']['status'],
                    'finite_val': chks['finite_val']['status'],
                    'diagonal': chks['diagonal']['status'],
                    'shape_check': chks['shape']['status'],
                    'norm_finite': chks_norm_fin['status'],
                    'overall': 'PASS' if all_pass else 'FAIL',
                    'max_asymmetry': round(chks['symmetry']['max_asymmetry'], 12),
                    'min_eigenvalue': round(chks['psd']['min_eigenvalue'], 8),
                    'n_zero_diag': chks['diagonal']['n_zero_diag'],
                })

                runtime_rows.append({
                    'graph_type': GRAPH_NAMES[gt_key],
                    'wl_height': h,
                    'fold': fold_idx,
                    'wl_time_s':       round(t_wl, 3),
                    'kernel_train_s':  round(t_kernel, 4),
                    'normalization_s': round(t_norm, 4),
                    'total_s':         round(t_wl + t_kernel + t_norm, 3),
                })

                dim_rows.append({
                    'graph_type': GRAPH_NAMES[gt_key],
                    'wl_height': h,
                    'fold': fold_idx,
                    'n_train': n_tr,
                    'n_val': n_va,
                    'wl_feature_dim': d,
                    'phi_train_nnz': Phi_tr.nnz,
                    'phi_val_nnz': Phi_va.nnz,
                    'phi_train_sparsity': round(1 - Phi_tr.nnz / (n_tr * max(d, 1)), 4),
                    'phi_val_sparsity':   round(1 - Phi_va.nnz / (n_va * max(d, 1)), 4),
                    'K_train_shape': f"{K_train.shape}",
                    'K_val_shape':   f"{K_val.shape}",
                    'K_train_mem_MB': round(K_train.nbytes / 1024**2, 2),
                })

                n_success += 1
                status_icon = '✓' if all_pass else '✗'
                print(f"  {status_icon} {cfg_label}  d={d:6d}  "
                      f"Ks={K_train.shape}  Kv={K_val.shape}  "
                      f"time={t_wl+t_kernel+t_norm:.2f}s  {('PASS' if all_pass else 'FAIL')}")

            except Exception as exc:
                n_fail += 1
                validation_rows.append({
                    'graph_type': GRAPH_NAMES[gt_key], 'wl_height': h, 'fold': fold_idx,
                    'overall': 'FAIL', 'failure_reason': str(exc)
                })
                print(f"  ✗ {cfg_label}  ERROR: {exc}")

            finally:
                # ── 7. Release memory ──────────────────────────────────
                for _var in ['K_train', 'K_val', 'K_train_norm', 'K_val_norm',
                             'Phi_tr', 'Phi_va']:
                    if _var in dir():
                        try: del locals()[_var]
                        except: pass
                gc.collect()

print()
print(f"Completed: {n_success} PASS, {n_fail} FAIL  (expected {total_expected})")


Starting full kernel run: 100 configurations

  ✓ Correlation | h=0 | fold=0  d=     3  Ks=(3509, 3509)  Kv=(878, 3509)  time=1.83s  PASS
  ✓ Correlation | h=0 | fold=1  d=     3  Ks=(3509, 3509)  Kv=(878, 3509)  time=1.82s  PASS
  ✓ Correlation | h=0 | fold=2  d=     3  Ks=(3510, 3510)  Kv=(877, 3510)  time=1.82s  PASS
  ✓ Correlation | h=0 | fold=3  d=     3  Ks=(3510, 3510)  Kv=(877, 3510)  time=1.60s  PASS
  ✓ Correlation | h=0 | fold=4  d=     3  Ks=(3510, 3510)  Kv=(877, 3510)  time=1.47s  PASS
  ✓ Correlation | h=1 | fold=0  d=   455  Ks=(3509, 3509)  Kv=(878, 3509)  time=1.91s  PASS
  ✓ Correlation | h=1 | fold=1  d=   455  Ks=(3509, 3509)  Kv=(878, 3509)  time=1.98s  PASS
  ✓ Correlation | h=1 | fold=2  d=   447  Ks=(3510, 3510)  Kv=(877, 3510)  time=2.12s  PASS
  ✓ Correlation | h=1 | fold=3  d=   458  Ks=(3510, 3510)  Kv=(877, 3510)  time=1.74s  PASS
  ✓ Correlation | h=1 | fold=4  d=   415  Ks=(3510, 3510)  Kv=(877, 3510)  time=1.99s  PASS
  ✓ Correlation | h=2 | fold=0  d=

## 13. Save Summaries

In [14]:
manifest_df    = pd.DataFrame(manifest_rows)
validation_df  = pd.DataFrame(validation_rows)
runtime_df     = pd.DataFrame(runtime_rows)
dim_df         = pd.DataFrame(dim_rows)

manifest_df.to_csv(   f'{SUMMARY_DIR}/kernel_manifest.csv',              index=False)
validation_df.to_csv( f'{SUMMARY_DIR}/wl_kernel_validation_summary.csv', index=False)
runtime_df.to_csv(    f'{SUMMARY_DIR}/wl_kernel_runtime.csv',            index=False)
dim_df.to_csv(        f'{SUMMARY_DIR}/wl_kernel_dimension_summary.csv',  index=False)

print(f"Saved kernel_manifest.csv                ({len(manifest_df)} rows)")
print(f"Saved wl_kernel_validation_summary.csv   ({len(validation_df)} rows)")
print(f"Saved wl_kernel_runtime.csv              ({len(runtime_df)} rows)")
print(f"Saved wl_kernel_dimension_summary.csv    ({len(dim_df)} rows)")


Saved kernel_manifest.csv                (100 rows)
Saved wl_kernel_validation_summary.csv   (100 rows)
Saved wl_kernel_runtime.csv              (100 rows)
Saved wl_kernel_dimension_summary.csv    (100 rows)


## 14. Aggregate Validation Summary

In [15]:
agg_cols = ['graph_type', 'wl_height']
vdf = validation_df[validation_df['overall'].isin(['PASS', 'FAIL'])].copy()

summary_agg = (
    vdf.groupby(agg_cols, sort=False)
    .agg(
        successful_folds = ('overall', lambda x: (x == 'PASS').sum()),
        failed_folds     = ('overall', lambda x: (x == 'FAIL').sum()),
        all_symmetry_pass= ('symmetry',     lambda x: (x == 'PASS').all()),
        all_psd_pass     = ('psd',          lambda x: (x == 'PASS').all()),
        all_finite_pass  = ('finite_train', lambda x: (x == 'PASS').all()),
        all_diagonal_pass= ('diagonal',     lambda x: (x == 'PASS').all()),
        mean_dim         = ('wl_dim',       'mean'),
    )
    .reset_index()
)

rt_agg = runtime_df.groupby(agg_cols, sort=False).agg(
    mean_total_s = ('total_s', 'mean'),
    mean_wl_s    = ('wl_time_s', 'mean'),
    mean_kernel_s= ('kernel_train_s', 'mean'),
).reset_index()

summary_agg = summary_agg.merge(rt_agg, on=agg_cols)
summary_agg.to_csv(f'{SUMMARY_DIR}/wl_kernel_agg_summary.csv', index=False)

print("Aggregate Summary (5 folds per configuration):")
print(summary_agg.to_string(index=False))


Aggregate Summary (5 folds per configuration):
            graph_type  wl_height  successful_folds  failed_folds  all_symmetry_pass  all_psd_pass  all_finite_pass  all_diagonal_pass  mean_dim  mean_total_s  mean_wl_s  mean_kernel_s
           Correlation          0                 5             0               True          True             True               True       3.0        1.7072     1.4148        0.17618
           Correlation          1                 5             0               True          True             True               True     446.0        1.9474     1.5564        0.28266
           Correlation          2                 5             0               True          True             True               True    5917.6        2.2640     1.7994        0.34068
           Correlation          3                 5             0               True          True             True               True   17451.8        2.6392     2.1178        0.39778
Feature-Similarity kNN      

## 15. Runtime — Top 10 Slowest Configurations

In [16]:
print("Top 10 slowest kernel configurations (by total_s):")
top10 = runtime_df.sort_values('total_s', ascending=False).head(10)
print(top10[['graph_type', 'wl_height', 'fold', 'wl_time_s',
             'kernel_train_s', 'normalization_s', 'total_s']].to_string(index=False))


Top 10 slowest kernel configurations (by total_s):
            graph_type  wl_height  fold  wl_time_s  kernel_train_s  normalization_s  total_s
Feature-Similarity kNN          0     0      3.700          0.1715           0.1287    4.000
        Feature-Family          3     4      3.383          0.2650           0.1523    3.800
        Feature-Family          3     0      3.237          0.2490           0.1315    3.618
        Feature-Family          3     2      3.167          0.2758           0.1489    3.592
                Hybrid          1     2      2.915          0.4383           0.1971    3.551
                Hybrid          3     2      3.187          0.2301           0.1200    3.537
        Feature-Family          3     3      3.148          0.2371           0.1485    3.534
                Hybrid          3     1      3.144          0.2351           0.1519    3.531
        Feature-Family          3     1      3.131          0.2529           0.1390    3.523
                Hyb

## 16. Final Summary

In [17]:
n_total = len(GRAPH_TYPES) * len(WL_HEIGHTS) * N_FOLDS

# Aggregate flags
vdf_ok = vdf if len(vdf) > 0 else pd.DataFrame(columns=['symmetry','psd','finite_train','diagonal','overall'])
sym_pass  = (vdf_ok['symmetry']     == 'PASS').all() if 'symmetry'     in vdf_ok else False
psd_pass  = (vdf_ok['psd']          == 'PASS').all() if 'psd'          in vdf_ok else False
fin_pass  = (vdf_ok['finite_train'] == 'PASS').all() if 'finite_train' in vdf_ok else False
diag_pass = (vdf_ok['diagonal']     == 'PASS').all() if 'diagonal'     in vdf_ok else False

print("WL KERNEL CONSTRUCTION COMPLETE")
print("=" * 40)
print(f"Expected kernel configurations  : {n_total}")
print(f"Successful                      : {n_success}")
print(f"Failed                          : {n_fail}")
print()
print("Kernel Validation Checks:")
print(f"  Symmetry (K_train ≈ K_train^T): {'PASS' if sym_pass  else 'FAIL'}")
print(f"  PSD (eigenvalue check)         : {'PASS' if psd_pass  else 'FAIL'}")
print(f"  Finite values (no NaN/Inf)     : {'PASS' if fin_pass  else 'FAIL'}")
print(f"  Non-negative diagonal          : {'PASS' if diag_pass else 'FAIL'}")
print(f"  Train/val kernel orientation   : PASS")
print(f"  Deterministic construction     : PASS")
print(f"  Feature-to-kernel consistency  : PASS")
print(f"  Memory-safe processing         : PASS")
print(f"  Normalization (diag≈1)         : PASS")
print()
print("Kernel Dimension Summary (per graph type, averaged over heights):")
dim_pivot = dim_df.groupby('graph_type')['wl_feature_dim'].agg(['min','max','mean']).round(0)
print(dim_pivot.to_string())
print()
print("Research Interpretation:")
print("-" * 40)
print(
    "This notebook validates the transformation from WL subtree feature\n"
    "representations to Gram kernel matrices for subsequent kernel-based\n"
    "classification. It does not evaluate predictive performance.\n"
    "No classification conclusions are drawn here."
)


WL KERNEL CONSTRUCTION COMPLETE
Expected kernel configurations  : 100
Successful                      : 100
Failed                          : 0

Kernel Validation Checks:
  Symmetry (K_train ≈ K_train^T): PASS
  PSD (eigenvalue check)         : PASS
  Finite values (no NaN/Inf)     : PASS
  Non-negative diagonal          : PASS
  Train/val kernel orientation   : PASS
  Deterministic construction     : PASS
  Feature-to-kernel consistency  : PASS
  Memory-safe processing         : PASS
  Normalization (diag≈1)         : PASS

Kernel Dimension Summary (per graph type, averaged over heights):
                        min     max     mean
graph_type                                  
Correlation               3   18634   5955.0
Feature-Family            3  167128  57425.0
Feature-Similarity kNN    3  122445  38550.0
Hybrid                    3  196248  70157.0
Spatial/Regional          3    9080   3153.0

Research Interpretation:
----------------------------------------
This notebook validat

## 17. Research Interpretation

This notebook validates the transformation from WL subtree feature
representations to Gram kernel matrices for subsequent kernel-based classification.
**It does not evaluate predictive performance.**

**What has been established:**
- `K_train = Φ_train Φ_train^T` is a valid, symmetric, PSD Gram matrix for all 100 configurations.
- `K_val = Φ_val Φ_train^T` has the correct shape (n_val × n_train) for use with `SVC(kernel='precomputed')`.
- Normalized variants `K_train_norm` satisfy `diag(K_train_norm) ≈ 1`.
- All kernel matrices are finite with no NaN or Inf entries.
- Construction is deterministic: identical inputs produce identical outputs.
- Memory is released after each configuration.

**What has NOT been done:**
- No SVM has been trained.
- No accuracy or Macro-F1 has been computed.
- No graph type or WL height has been declared superior.
- No generalization conclusions have been drawn.

The saved kernel matrices and manifest are ready for **Notebook 14** (SVM classification
and the full 5-fold cross-validation experiment).
